In [ ]:
# --- Install the PyTorch stack ---
print("Installing torch, torchvision, torchaudio...")
!pip install torch torchvision torchaudio

# --- Install other required Python libraries, UPGRADING Gradio ---
print("Installing transformers, accelerate, soundfile, librosa, gradio...")
!pip install transformers accelerate soundfile librosa gradio --upgrade

# --- Install FFmpeg ---
print("Installing FFmpeg...")
!apt-get update -qq
!apt-get install -qq ffmpeg

print("\nInstallation steps complete.")

In [ ]:
import gradio as gr
import torch
from transformers import pipeline
import os
import gc
import librosa
import soundfile as sf
from functools import lru_cache
import time

# 1. Tắt các cảnh báo và kiểm tra API không cần thiết
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

MODEL_NAME = "vinai/PhoWhisper-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK_LENGTH_S = 30

print(f"Running on: {DEVICE}")

@lru_cache(maxsize=None)
def load_model(model_source):
    # 2. Dọn dẹp bộ nhớ GPU trước khi load
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    print(f"Loading model: {model_source}...")
    try:
        # 3. Load mô hình với kiểu dữ liệu float16 để tiết kiệm VRAM
        # Nếu GPU của bạn đời mới (RTX 30xx trở lên), có thể dùng torch.bfloat16
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32

        pipe = pipeline(
            "automatic-speech-recognition",
            model=model_source,
            chunk_length_s=CHUNK_LENGTH_S,
            device=DEVICE,
            torch_dtype=dtype, # QUAN TRỌNG: Giảm 50% VRAM
            model_kwargs={
                "use_safetensors": False,
                "low_cpu_mem_usage": True # Tiết kiệm RAM hệ thống
            }
        )
        print("Model loaded successfully.")
        return pipe
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

asr_pipeline = load_model(MODEL_NAME)

# Các hàm phụ trợ (get_audio_duration, transcribe_audio_for_blocks, v.v.) giữ nguyên như cũ của bạn...
def get_audio_duration(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return 0
    try:
        duration = librosa.get_duration(path=audio_path)
        return duration
    except Exception as e:
        try:
            with sf.SoundFile(audio_path) as f:
                return len(f) / f.samplerate
        except:
            return 0

def transcribe_audio_for_blocks(audio_file_obj, progress=gr.Progress()):
    if asr_pipeline is None:
        return "Error: Model not loaded.", "Status: Error"
    if audio_file_obj is None:
        return "Please upload an audio file.", "Status: Waiting"

    duration = get_audio_duration(audio_file_obj)
    duration_str = f"{duration:.2f}s"
    progress(0, desc=f"Transcribing {duration_str}...")

    start_time = time.time()
    try:
        # ignore_warning=True giúp ẩn cảnh báo experimental chunking
        transcription_result = asr_pipeline(audio_file_obj, ignore_warning=True)
        processing_time = time.time() - start_time
        return transcription_result["text"], f"Done in {processing_time:.2f}s"
    except Exception as e:
        return str(e), "Failed"

def prepare_download_file(transcript_text):
    filename = "transcript.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(transcript_text)
    return filename

# --- UI Interface ---
# FIX: Bỏ theme ở đây để tránh Warning trên Gradio 6.0
with gr.Blocks(title="PhoWhisper ASR") as interface:
    gr.Markdown("# PhoWhisper ASR")

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath", label="Upload Vietnamese Audio File")
            transcribe_button = gr.Button("Transcribe Audio", variant="primary")

        with gr.Column():
            transcription_output_textbox = gr.Textbox(
                label="Transcription",
                lines=10,
                interactive=True,
                placeholder="Transcription will appear here...",
                # FIX: Tạm thời bỏ show_copy_button nếu lỗi version 6.0,
                # hoặc thay bằng copyable=True nếu bản Gradio bạn dùng hỗ trợ
            )
            status_textbox = gr.Textbox(label="Status", interactive=False)
            download_button = gr.Button("Prepare Download File")
            download_file_output = gr.File(label="Download your transcript")

    transcribe_button.click(
        fn=transcribe_audio_for_blocks,
        inputs=[audio_input],
        outputs=[transcription_output_textbox, status_textbox]
    )

    download_button.click(
        fn=prepare_download_file,
        inputs=[transcription_output_textbox],
        outputs=[download_file_output]
    )

if __name__ == "__main__":
    if asr_pipeline is None:
        print("Failed to load ASR pipeline.")
    else:
        # FIX: Chuyển theme xuống đây theo yêu cầu của Gradio 6.0
        interface.launch(share=True, theme=gr.themes.Soft())